


#**Detection and Classification**

This notebook has the ability to detect up to 4 different hand signs in an image, assigns each hand to a player using specified upright/upside-down camera arrangmenet, and classifies each hand as rock, paper, or scissors.

**Setting up all neccessary libraries**





In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp

print("Python:", sys.version.split()[0])
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)

**Setting up the camera, making all neccessary assumptions, and inputting images**

In [ ]:
IMAGE_PATH = "images/test_1.jpg"  # Change this to one of your five sample images

def load_image(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Image not found: {path}. Create an images folder beside this notebook "
            "and place the image there, then update IMAGE_PATH."
        )
    image = cv2.imread(path)
    if image is None:
        raise ValueError(f"OpenCV could not read image: {path}")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

**Hand Detection and Feature Extraction**

In [ ]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

def detect_hand_landmarks(image_rgb, max_num_hands=4,
                          min_detection_confidence=0.5,
                          min_tracking_confidence=0.5):
    """Return MediaPipe results for up to four hands in an RGB image."""
    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=max_num_hands,
        model_complexity=1,
        min_detection_confidence=min_detection_confidence,
        min_tracking_confidence=min_tracking_confidence
    ) as hands:
        return hands.process(image_rgb)

**Gesture/Orientation Classification**

In [ ]:
def joint_angle(a, b, c):
    """Angle ABC in degrees, where a, b, c are 2D or 3D points."""
    ba = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    bc = np.asarray(c, dtype=float) - np.asarray(b, dtype=float)
    denom = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denom == 0:
        return 0.0
    cosine = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def landmark_xyz(hand_landmarks, index):
    p = hand_landmarks.landmark[index]
    return np.array([p.x, p.y, p.z], dtype=float)


# MediaPipe landmark indices for index, middle, ring, and little fingers:
# (MCP, PIP, DIP, TIP)
FINGER_LANDMARKS = {
    "index":  (5, 6, 7, 8),
    "middle": (9, 10, 11, 12),
    "ring":   (13, 14, 15, 16),
    "little": (17, 18, 19, 20),
}

def classify_fingers(hand_landmarks, extension_angle_threshold=155):
    """Return a dictionary of extension flags and the resulting RPS label."""
    extended = {}
    angles = {}
    for finger, (mcp, pip, dip, tip) in FINGER_LANDMARKS.items():
        angle = joint_angle(
            landmark_xyz(hand_landmarks, mcp),
            landmark_xyz(hand_landmarks, pip),
            landmark_xyz(hand_landmarks, dip)
        )
        angles[finger] = angle
        extended[finger] = angle >= extension_angle_threshold

    pattern = tuple(extended[name] for name in
                    ("index", "middle", "ring", "little"))

    if pattern == (False, False, False, False):
        label = "Rock"
    elif pattern == (True, True, True, True):
        label = "Paper"
    elif pattern == (True, True, False, False):
        label = "Scissors"
    else:
        label = "Unknown"

    return label, extended, angles


def assign_player(hand_landmarks, vertical_threshold=0.015):
    """Assign player from wrist-to-middle-MCP vertical direction.

    Returns (player label, orientation label). Upright = Player 1;
    upside-down = Player 2. Ambiguous orientations are left unassigned.
    """
    wrist = landmark_xyz(hand_landmarks, 0)
    middle_mcp = landmark_xyz(hand_landmarks, 9)
    dy = middle_mcp[1] - wrist[1]  # image y increases downward

    if dy < -vertical_threshold:
        return "Player 1 (self)", "Upright"
    if dy > vertical_threshold:
        return "Player 2 (opponent)", "Upside-down"
    return "Unknown", "Ambiguous"

**Image Processing**

In [ ]:
def process_image(image_rgb, draw=True):
    results = detect_hand_landmarks(image_rgb)
    records = []
    annotated = image_rgb.copy()

    if results.multi_hand_landmarks:
        height, width = image_rgb.shape[:2]
        for i, hand_lms in enumerate(results.multi_hand_landmarks):
            player, orientation = assign_player(hand_lms)
            gesture, finger_flags, angles = classify_fingers(hand_lms)

            xs = [lm.x for lm in hand_lms.landmark]
            ys = [lm.y for lm in hand_lms.landmark]
            bbox = (
                max(0, int(min(xs) * width)),
                max(0, int(min(ys) * height)),
                min(width - 1, int(max(xs) * width)),
                min(height - 1, int(max(ys) * height))
            )

            records.append({
                "detection": i + 1,
                "player": player,
                "orientation": orientation,
                "gesture": gesture,
                "finger_flags": finger_flags,
                "angles_degrees": angles,
                "bbox_xyxy": bbox
            })

            if draw:
                mp_drawing.draw_landmarks(
                    annotated, hand_lms, mp_hands.HAND_CONNECTIONS
                )
                x1, y1, x2, y2 = bbox
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 200, 0), 2)
                text = f"{player}: {gesture}"
                cv2.putText(annotated, text, (x1, max(25, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 0, 0), 2)

    return records, annotated


def display_results(image_rgb, records, annotated):
    plt.figure(figsize=(12, 7))
    plt.imshow(annotated)
    plt.axis("off")
    plt.title(f"Detected hands: {len(records)} (expected: 4)")
    plt.show()

    if not records:
        print("No hands detected.")
        return

    print("Detection | Player | Orientation | Gesture | Bounding box")
    for r in records:
        print(f'{r["detection"]:9} | {r["player"]:21} | '
              f'{r["orientation"]:12} | {r["gesture"]:7} | {r["bbox_xyxy"]}')

**Results Displayed**

In [ ]:
image_rgb = load_image(IMAGE_PATH)
detections, annotated_image = process_image(image_rgb)
display_results(image_rgb, detections, annotated_image)

**Testing Demonstration Images**

In [ ]:
SAMPLE_IMAGE_PATHS = [
    "images/test_1.jpg",
    "images/test_2.jpg",
    "images/test_3.jpg",
    "images/test_4.jpg",
    "images/test_5.jpg",
]

# Replace None with manually verified expected detections for each image.
# Example: [{"player": "Player 1 (self)", "gesture": "Rock"}, ...]
GROUND_TRUTH = {
    path: None for path in SAMPLE_IMAGE_PATHS
}

sample_results = {}
for path in SAMPLE_IMAGE_PATHS:
    print("\\n" + "=" * 80)
    print("IMAGE:", path)
    try:
        img = load_image(path)
        recs, ann = process_image(img)
        sample_results[path] = recs
        display_results(img, recs, ann)
    except (FileNotFoundError, ValueError) as error:
        print(error)

**Quantitative Evaluation**

In [ ]:
def classification_metrics(y_true, y_pred, labels):
    """Compute accuracy and per-class counts without requiring scikit-learn."""
    y_true = list(y_true)
    y_pred = list(y_pred)
    if len(y_true) != len(y_pred):
        raise ValueError("y_true and y_pred must have the same length.")
    correct = sum(a == b for a, b in zip(y_true, y_pred))
    accuracy = correct / len(y_true) if y_true else float("nan")
    confusion = {
        actual: {pred: 0 for pred in labels}
        for actual in labels
    }
    for actual, pred in zip(y_true, y_pred):
        if actual in confusion and pred in confusion[actual]:
            confusion[actual][pred] += 1
    return accuracy, confusion


def plot_confusion_matrix(confusion, labels, title):
    matrix = np.array([[confusion[a][p] for p in labels] for a in labels])
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix)
    ax.set_xticks(range(len(labels)), labels=labels)
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(matrix[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
    return matrix

**Labelled Test Results**

In [ ]:
GESTURE_LABELS = ["Rock", "Paper", "Scissors", "Unknown"]
PLAYER_LABELS = ["Player 1 (self)", "Player 2 (opponent)", "Unknown"]

# Replace these empty lists with labels from your test set.
true_gestures = []
predicted_gestures = []
true_players = []
predicted_players = []

if true_gestures:
    gesture_acc, gesture_cm = classification_metrics(
        true_gestures, predicted_gestures, GESTURE_LABELS
    )
    print(f"Gesture classification accuracy: {gesture_acc:.3f}")
    plot_confusion_matrix(gesture_cm, GESTURE_LABELS,
                          "Gesture classification confusion matrix")
else:
    print("Enter manually labelled gesture results to calculate accuracy.")

if true_players:
    player_acc, player_cm = classification_metrics(
        true_players, predicted_players, PLAYER_LABELS
    )
    print(f"Player assignment accuracy: {player_acc:.3f}")
    plot_confusion_matrix(player_cm, PLAYER_LABELS,
                          "Player assignment confusion matrix")
else:
    print("Enter manually labelled player results to calculate accuracy.")